# NOROM Bimanual - raw veri kesif notebooku

Bu notebook `Data/Raw/*/N.mat` dosyalarini MATLAB olmadan okur.

Dosyalar MATLAB v5 formatinda ama icindeki tek degisken `out`, bir
`Simulink.SimulationOutput` nesnesi (MCOS class). `scipy.io.loadmat` bunu
dogrudan cozemez; veri dosyanin *function workspace* bolumunde saklaniyor.
`src/simout.py` o bolumu acip logged signal'lari numpy array olarak veriyor.


In [ ]:
import sys, os, glob
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simout

RAW = os.path.abspath('../Data/Raw')
sessions = sorted(os.listdir(RAW))
sessions

## 1. Bir trial ac ve icinde ne var bak

In [ ]:
tr = simout.load(os.path.join(RAW, '19.08_1', '1.mat'))
tr.meta['ModelName'], tr.meta['WallClockTimestampStart'], tr.meta['StopTime']

In [ ]:
desc = tr.describe()
pd.set_option('display.max_rows', 200, 'display.width', 200)
desc

### Sinyal sozlugu (ilk okumadan cikan yorum)

- `tout` : zaman, 0-70 s, 1 kHz -> 70001 ornek. Her trial sabit 70 s.
- `leader_x`, `leader_y` (= `Leader` bloku) : takip edilecek referans yolu.
  **Butun trial'larda birebir ayni** - Lissajous benzeri kapali bir egri.
- `F1_PosX/F1_PosY/F1_Phi` : 1. oyuncunun aracinin x, y, yonelim (psi).
- `F2_PosX/F2_PosY/F2_Phi` : 2. oyuncunun araci.
- `X1_F1`, `X2_F1` : 1. oyuncunun iki joystick ekseni (Joystick 4), [-1, 1].
- `X1_F2`, `X2_F2` : 2. oyuncunun iki joystick ekseni (Joystick 3).
- `L_Acc_F1`, `R_Acc_F1` : 1. oyuncunun sol/sag palet ivmesi, [-7, 7] (tank surusu).
- `L_Acc_F2`, `R_Acc_F2` : 2. oyuncu icin ayni.
- `T_human_F1`, `T_Human_F2` : insandan gelen tork/komut. F1'de [-4.2, 4.2],
  F2'de [-2.1, 2.1] -> iki oyuncuda farkli gain var gibi, kontrol edilmeli.
- `Score`, `Score2` : monoton artan puan sayaclari (F1 ve F2).
- `ScopeData2`, `ScopeData13` (blok adi `Stochastic`, `Stochastic1`) : 0/1 ikili
  sinyal, trial basina ~8-10 kez aciliyor, toplam surenin ~%17'si. Haptik
  olayla / bozucuyla ilgili olabilir.
- `Force_Feedback*`, `Noise*`, `Left_x`, `Right_x` : bu dosyalarda hep sabit 0.

### Dikkat: isim tuzagi

Ayni sinyal iki isimle loglanmis ve **son ekler capraz**:

| kisa isim | esiti |
|---|---|
| `follower_x1`, `Follower_y1`, `Followerpsi1` | `F2_PosX`, `F2_PosY`, `F2_Phi` |
| `follower_x2`, `Follower_y2`, `Followerpsi2` | `F1_PosX`, `F1_PosY`, `F1_Phi` |
| `Left_Acc1`, `Right_Acc1`, `R_heading1` | `L_Acc_F2`, `R_Acc_F2`, `R_heading_F2` |
| `Left_Acc2`, `Right_Acc2`, `R_heading2` | `L_Acc_F1`, `R_Acc_F1`, `R_heading_F1` |

Yani `...1` son eki F2'ye, `...2` son eki F1'e gidiyor. Karisikligi onlemek icin
sadece `F1_*` / `F2_*` isimlerini kullanmak daha guvenli.

In [ ]:
# yukaridaki esitligi dogrula
for a, b in [('follower_x1','F2_PosX'), ('follower_x2','F1_PosX'),
             ('Left_Acc1','L_Acc_F2'), ('Left_Acc2','L_Acc_F1'),
             ('R_heading1','R_heading_F2'), ('R_heading2','R_heading_F1')]:
    print(f'{a:14s} == {b:14s} :', np.allclose(tr.get(a), tr.get(b)))

## 2. Tek trial gorsellestirme

In [ ]:
def plot_trial(tr, title=''):
    t = tr.time()
    lx, ly = tr.get('leader_x'), tr.get('leader_y')
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))

    ax[0].plot(lx, ly, 'k-', lw=1.2, label='leader')
    for F, c in [('F1', 'tab:blue'), ('F2', 'tab:orange')]:
        ax[0].plot(tr.get(f'{F}_PosX'), tr.get(f'{F}_PosY'), lw=.7, alpha=.8, color=c, label=F)
    ax[0].set_aspect('equal'); ax[0].legend(fontsize=8); ax[0].set_title('XY yol')

    for F, c in [('F1', 'tab:blue'), ('F2', 'tab:orange')]:
        e = np.hypot(tr.get(f'{F}_PosX') - lx, tr.get(f'{F}_PosY') - ly)
        ax[1].plot(t, e, lw=.6, color=c, label=f'{F} RMSE={np.sqrt((e**2).mean()):.2f}')
    ax[1].legend(fontsize=8); ax[1].set_xlabel('s'); ax[1].set_title('takip hatasi')

    ax[2].plot(t, tr.get('X1_F1'), lw=.6, label='F1 stick 1')
    ax[2].plot(t, tr.get('X2_F1'), lw=.6, label='F1 stick 2')
    ax[2].legend(fontsize=8); ax[2].set_xlabel('s'); ax[2].set_title('joystick')

    fig.suptitle(title); fig.tight_layout()

plot_trial(tr, '19.08_1 / trial 1')

## 3. Trial'lar arasi ogrenme egrisi

Deney tasarimi (soylenene gore): her oturumda 15 trial, **ilk 3 ve son 3**
haptik geri bildirim yok, aradaki 9 trial'da var. Yani baseline -> adaptasyon
-> washout / after-effect deseni.

Asagidaki hucre bir oturumun 15 trial'ini okuyup ozet metrik cikarir.
15 x 20 MB oldugu icin biraz surer; sonucu diske cache'liyoruz.

In [ ]:
def trial_summary(path):
    tr = simout.load(path)
    t = tr.time()
    lx, ly = tr.get('leader_x'), tr.get('leader_y')
    row = {'file': os.path.basename(path)}
    for F in ('F1', 'F2'):
        e = np.hypot(tr.get(f'{F}_PosX') - lx, tr.get(f'{F}_PosY') - ly)
        row[f'{F}_rmse'] = float(np.sqrt((e**2).mean()))
        row[f'{F}_med'] = float(np.median(e))
    row['Score'] = float(tr.get('Score')[-1])
    row['Score2'] = float(tr.get('Score2')[-1])
    row['stoch_frac'] = float(tr.get('ScopeData2').mean())
    row['ff_max'] = float(np.abs(tr.get('Force_Feedback')).max())
    row['t_start'] = str(tr.meta.get('WallClockTimestampStart'))
    return row


def session_summary(session, cache_dir='../Data/derived'):
    os.makedirs(cache_dir, exist_ok=True)
    cache = os.path.join(cache_dir, f'summary_{session}.csv')
    if os.path.exists(cache):
        return pd.read_csv(cache)
    files = sorted(glob.glob(os.path.join(RAW, session, '*.mat')),
                   key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))
    df = pd.DataFrame([trial_summary(p) for p in files])
    df.insert(0, 'trial', range(1, len(df) + 1))
    df.insert(0, 'session', session)
    df.to_csv(cache, index=False)
    return df


s1 = session_summary('19.08_1')
s1

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(s1.trial, s1.F1_rmse, 'o-', label='F1')
ax[0].plot(s1.trial, s1.F2_rmse, 'o-', label='F2')
for x in (3.5, 12.5):
    ax[0].axvline(x, ls='--', c='k', lw=.8)
ax[0].set_xlabel('trial'); ax[0].set_ylabel('takip RMSE'); ax[0].legend()
ax[0].set_title('haptik: trial 4-12 arasi (varsayim)')

ax[1].plot(s1.trial, s1.Score, 'o-', label='F1')
ax[1].plot(s1.trial, s1.Score2, 'o-', label='F2')
ax[1].set_xlabel('trial'); ax[1].set_ylabel('final score'); ax[1].legend()
fig.tight_layout()

## 4. Acik sorular

1. `Force_Feedback` sinyali baktigimiz iki trial'da da sabit 0. Haptik komut
   baska bir sinyalde mi (`T_human_*`?) yoksa hic loglanmadi mi?
2. `Stochastic` / `Stochastic1` ikili sinyali ne? Bozucu (perturbation) tetigi
   olabilir gibi duruyor.
3. `T_human_F1` [-4.2, 4.2] ama `T_Human_F2` [-2.1, 2.1]. Iki oyuncuda kasitli
   gain farki mi, yoksa model hatasi mi?
4. `Score` neye gore artiyor? Leader'a yakinlikla mi, toplanan hedefle mi?
5. Bir oturumdaki 15 trial ayni ciftin mi, yoksa oturum = cift mi?
